# Polynomial Proxy Labels for Spend-Position Evaluation

This notebook is intentionally separate from the Beta CDF model notebook. It creates retrospective proxy labels from complete historical clustered cumulative spend data and writes `clustered_curve_proxy_labels.csv`. The model notebook consumes that labeled file for ROC/AUC and threshold-sweep analysis.

## Method

The proxy reference is an anchored polynomial fitted on the training rows only. Anchoring forces the curve through `(0, 0)` and `(1, 1)`:

```text
P(x) = c0*x + x*(1-x)*(c1 + c2*x + ...)
```

The fitted curve is clipped to `[0, 1]` before labels are generated. A row is labeled fast when actual cumulative spend is more than the threshold above the polynomial reference, and slow when it is more than the threshold below it.

## Output Summary

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_adams.csv |
| Output labeled CSV | clustered_curve_proxy_labels_adams.csv |
| Rows labeled | 606 |
| Train rows | 414 |
| Test rows | 192 |
| Selected polynomial degree | 4 |
| Label threshold | 15.00% |
| Fast proxy positives on test | 19 |
| Slow proxy positives on test | 43 |

## Polynomial Fit Diagnostics

| Degree | Selected | Train MAE | Train RMSE | Test MAE | Test RMSE | Train bias | Clip share | Monotonic violations | Coefficients |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 3 | no | 0.1194 | 0.1691 | 0.1193 | 0.1538 | -0.0008 | 0.0000 | 0 | 0.994980, 0.135912, 0.152961 |
| 4 | yes | 0.1194 | 0.1691 | 0.1194 | 0.1538 | -0.0007 | 0.0000 | 0 | 0.994775, 0.141316, 0.128017, 0.026884 |

## Reference Curve Plot



## Proxy Label Plot



In [ ]:
import csv
from pathlib import Path

label_path = Path('clustered_curve_proxy_labels.csv')
with label_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(label_path)
print(len(rows))
print(reader.fieldnames)
print('fast labels:', sum(int(r['fast_proxy_label']) for r in rows if r['split'] == 'test'))
print('slow labels:', sum(int(r['slow_proxy_label']) for r in rows if r['split'] == 'test'))

## Notes for Downstream Modeling

These labels are proxy labels, not true budget-overrun or schedule-overrun outcomes. They are still useful because they come from a reference curve fit to complete historical behavior and are separated from the Beta CDF and linear models being evaluated downstream.